# Kaggle Model Inference API - Ultra-Robust Version

**Fixes for slow startup:**
- Check/kill port 8001 if in use
- Longer timeout (2 min instead of 1 min)
- Better error messages
- Simpler async pattern

In [1]:

import subprocess
import logging

print("🔍 Checking for processes on port 8001...")
try:
    result = subprocess.run(
        ["lsof", "-ti", ":8001"],
        capture_output=True,
        text=True,
        timeout=5
    )
    if result.stdout.strip():
        pids = result.stdout.strip().split('\n')
        for pid in pids:
            try:
                subprocess.run(["kill", "-9", pid], timeout=2)
                print(f"   ✅ Killed process {pid}")
            except:
                pass
except Exception as e:
    print(f"   ℹ️  Could not check processes: {e}")


print("📦 Installing dependencies...")
!pip install -q fastapi uvicorn pyngrok transformers torch accelerate nest-asyncio
print("   ✅ Installed")

🔍 Checking for processes on port 8001...
📦 Installing dependencies...
   ✅ Installed


In [2]:

import nest_asyncio
import logging
import sys

nest_asyncio.apply()
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    stream=sys.stdout
)
logger = logging.getLogger(__name__)

print("✅ Async event loop configured")

✅ Async event loop configured


In [3]:

from functools import lru_cache
from fastapi import FastAPI
from pydantic import BaseModel
import logging

logger = logging.getLogger(__name__)

app = FastAPI(title="Kaggle Model Inference API", version="1.0.0")

class GenerationRequest(BaseModel):
    system_prompt: str
    user_content: str
    generation_options: dict = {}

@lru_cache(maxsize=1)
def load_generator():
    """Load model once and cache it."""
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    logger.info("📥 Loading Qwen3-4B model...")
    model_id = "Qwen/Qwen3-4B-Instruct-2507"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map={"": 0},   
    attn_implementation="sdpa",
    )
  
    logger.info("✅ Model loaded successfully")
    return tokenizer, model

@app.get("/health")
def health() -> dict[str, str]:
    return {"status": "ok", "service": "kaggle-model"}

@app.post("/generate")
def generate(request: GenerationRequest) -> dict[str, str]:
    import torch
    try:
        tokenizer, model = load_generator()
        messages = [
            {"role": "system", "content": request.system_prompt},
            {"role": "user", "content": request.user_content},
        ]
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        input_length = inputs.input_ids.shape[1]

       
        options = {
            "max_new_tokens": 4096,
            "do_sample": True,
            "temperature": 0.3,
            "top_p": 0.9,
        }
        options.update(request.generation_options)

        logger.info(f"🔄 Generating with {options['max_new_tokens']} tokens...")
        with torch.inference_mode():
            outputs = model.generate(**inputs, **options)
        text = tokenizer.decode(
            outputs[0][input_length:],
            skip_special_tokens=True,
        )
        logger.info(f"✅ Generated {len(text)} characters")
        return {"text": text}
    except Exception as e:
        logger.error(f"❌ Generation failed: {str(e)}", exc_info=True)
        raise

print("✅ FastAPI app configured")

✅ FastAPI app configured


In [4]:
# CELL 3: Setup ngrok
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
import logging

logger = logging.getLogger(__name__)

try:
    user_secrets = UserSecretsClient()
    token = user_secrets.get_secret("TOKEN")
    ngrok.set_auth_token(token)
    logger.info("✅ ngrok token configured")
except Exception as e:
    logger.warning(f"⚠️  ngrok token not found: {e}. Using free tier.")

2026-09-13 20:42:47,401 - INFO - Updating authtoken for default "config_path" of "ngrok_path": /root/.config/ngrok/ngrok
2026-09-13 20:42:47,434 - INFO - ✅ ngrok token configured


In [5]:
# CELL 4: Start server with robust error handling
import asyncio
import logging
import time
from pyngrok import ngrok
import uvicorn

logger = logging.getLogger(__name__)

class SimpleUvicornServer:
    """Simple wrapper for running Uvicorn in async context."""
    def __init__(self, app, host="0.0.0.0", port=8001):
        self.app = app
        self.host = host
        self.port = port
        self.server = None
        
    async def start(self):
        logger.info(f"🚀 Starting FastAPI server on {self.host}:{self.port}...")
        config = uvicorn.Config(
            self.app,
            host=self.host,
            port=self.port,
            log_level="info"
        )
        self.server = uvicorn.Server(config)
     
        return asyncio.create_task(self.server.serve())

async def wait_for_port(host="127.0.0.1", port=8001, timeout=120):
    """Wait for port to be accessible. Timeout = 2 minutes for model load."""
    import socket
    
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            sock.settimeout(1)
            result = sock.connect_ex((host, port))
            sock.close()
            if result == 0:
                logger.info(f"✅ Port {port} is accessible!")
                return True
        except:
            pass
        
        elapsed = time.time() - start_time
        if elapsed % 10 < 1:  # Log every 10s
            logger.info(f"   Waiting for server... ({int(elapsed)}s elapsed)")
        
        await asyncio.sleep(1)
    
    raise RuntimeError(
        f"Server did not start on {host}:{port} within {timeout}s.\n"
        f"Kaggle might be loading the model. Try restarting the kernel."
    )

async def main():
    # Start server
    server_wrapper = SimpleUvicornServer(app, host="0.0.0.0", port=8001)
    server_task = await server_wrapper.start()
    
 
    logger.info("⏳ Waiting for server to be ready (this can take 30-60s for first load)...")
    await wait_for_port(timeout=120)  # 2 minute timeout
    
 
    logger.info("🌐 Starting ngrok tunnel...")
    try:
        public_url = ngrok.connect(8001, "http").public_url
        logger.info(f"✅ ngrok tunnel created: {public_url}")
    except Exception as e:
        logger.error(f"❌ ngrok failed: {e}")
        raise
    
    return server_task, public_url


print("\n" + "="*70)
print("🎯 STARTING SERVER...")
print("="*70)
server_task, public_url = await main()
print("\n" + "="*70)
print(f"✅ READY! Model API at: {public_url}")
print(f"   Health: {public_url}/health")
print(f"   Generate: {public_url}/generate (POST)")
print("="*70)


🎯 STARTING SERVER...
2026-09-13 20:42:47,541 - INFO - 🚀 Starting FastAPI server on 0.0.0.0:8001...
2026-09-13 20:42:47,543 - INFO - ⏳ Waiting for server to be ready (this can take 30-60s for first load)...
2026-09-13 20:42:47,544 - INFO -    Waiting for server... (0s elapsed)


INFO:     Started server process [59]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


2026-09-13 20:42:48,545 - INFO - ✅ Port 8001 is accessible!
2026-09-13 20:42:48,546 - INFO - 🌐 Starting ngrok tunnel...
2026-09-13 20:42:48,547 - INFO - Opening tunnel named: http-8001-9a85502a-91e2-42f0-b548-b9bd5d61995d
2026-09-13 20:42:48,567 - INFO - t=2026-09-13T20:42:48+0000 lvl=info msg="no configuration paths supplied"
2026-09-13 20:42:48,568 - INFO - t=2026-09-13T20:42:48+0000 lvl=info msg="using configuration at default config path" path=/root/.config/ngrok/ngrok.yml
2026-09-13 20:42:48,568 - INFO - t=2026-09-13T20:42:48+0000 lvl=info msg="open config file" path=/root/.config/ngrok/ngrok.yml err=<nil>
2026-09-13 20:42:48,867 - INFO - t=2026-09-13T20:42:48+0000 lvl=info msg="FIPS 140 mode" enabled=false
2026-09-13 20:42:48,876 - INFO - t=2026-09-13T20:42:48+0000 lvl=info msg="starting web service" obj=web addr=127.0.0.1:4040 allow_hosts=[]
2026-09-13 20:42:49,214 - INFO - t=2026-09-13T20:42:49+0000 lvl=info msg="client session established" obj=tunnels.session
2026-09-13 20:42:

In [6]:
# CELL 5: Health monitoring (optional)
import asyncio
import logging
from datetime import datetime

logger = logging.getLogger(__name__)

async def monitor_server(interval=30):
    """Monitor server health periodically."""
    import httpx
    
    while True:
        try:
            async with httpx.AsyncClient(timeout=5) as client:
                response = await client.get(f"{public_url}/health")
                if response.status_code == 200:
                    logger.info(f"[{datetime.now().strftime('%H:%M:%S')}] Server healthy ✅")
                else:
                    logger.warning(f"[{datetime.now().strftime('%H:%M:%S')}] Server returned {response.status_code}")
        except Exception as e:
            logger.error(f"[{datetime.now().strftime('%H:%M:%S')}] Health check failed: {e}")
        
        await asyncio.sleep(interval)

monitor_task = asyncio.create_task(monitor_server())
print("✅ Health monitoring started (every 30s)")

✅ Health monitoring started (every 30s)


## ✅ Server Running
    
    Your model API is now available at the URL above.
    
    **To test:**
    ```bash
    curl -X POST https://YOUR-NGROK-URL/generate \
      -H "Content-Type: application/json" \
      -d '{"system_prompt":"helpful","user_content":"say hello"}'
    ```
    
    **Keep this notebook running** - the server will stay up as long as the kernel is active.
    
    If you get an error, check:
    1. Kernel is still running
    2. ngrok tunnel is active
    3. Model is fully loaded (watch the logs above)
   

In [8]:
!nvidia-smi

Sun Sep 13 20:47:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   68C    P0             67W /   70W |    8537MiB /  15360MiB |     75%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


2026-09-13 20:49:26,037 - INFO - t=2026-09-13T20:49:26+0000 lvl=info msg="join connections" obj=join id=9464d0883ddd l=127.0.0.1:8001 r=34.12.96.251:59098
INFO:     34.12.96.251:0 - "GET /health HTTP/1.1" 200 OK
2026-09-13 20:49:26,141 - INFO - HTTP Request: GET https://162f-34-12-96-251.ngrok-free.app/health "HTTP/1.1 200 OK"
2026-09-13 20:49:26,143 - INFO - [20:49:26] Server healthy ✅
2026-09-13 20:49:56,427 - INFO - t=2026-09-13T20:49:56+0000 lvl=info msg="join connections" obj=join id=f919182a4bb2 l=127.0.0.1:8001 r=34.12.96.251:46018
INFO:     34.12.96.251:0 - "GET /health HTTP/1.1" 200 OK
2026-09-13 20:49:56,528 - INFO - HTTP Request: GET https://162f-34-12-96-251.ngrok-free.app/health "HTTP/1.1 200 OK"
2026-09-13 20:49:56,529 - INFO - [20:49:56] Server healthy ✅
2026-09-13 20:50:26,804 - INFO - t=2026-09-13T20:50:26+0000 lvl=info msg="join connections" obj=join id=2627b1ff5cbe l=127.0.0.1:8001 r=34.12.96.251:47766
INFO:     34.12.96.251:0 - "GET /health HTTP/1.1" 200 OK
2026-09-